# Day 07 — Clinical Operations Dashboard

**Dataset:** Healthcare Analytics II (318k+ patient records, 31 hospitals)

**Goal:** Analyse length of stay, admission types and hospital benchmarking

**Tools:** Python, Pandas, Plotly

**Relevant to:** NHS, Optum, Health Catalyst analyst roles

In [1]:
# Importing Libraries

import os
os.chdir(os.path.expanduser('~/Desktop/DataAnalystJourney/day07-clinical-operations'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✓ Libraries loaded")

✓ Libraries loaded


In [3]:
# Loading and inspecting dataset

df = pd.read_csv('data/train_data.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum()>0]}")
print(f"\nAdmission types: {df['Type of Admission'].unique()}")
print(f"\nStay values: {df['Stay'].unique()}")


Shape: (318438, 18)

Columns: ['case_id', 'Hospital_code', 'Hospital_type_code', 'City_Code_Hospital', 'Hospital_region_code', 'Available Extra Rooms in Hospital', 'Department', 'Ward_Type', 'Ward_Facility_Code', 'Bed Grade', 'patientid', 'City_Code_Patient', 'Type of Admission', 'Severity of Illness', 'Visitors with Patient', 'Age', 'Admission_Deposit', 'Stay']

Missing values:
Bed Grade             113
City_Code_Patient    4532
dtype: int64

Admission types: ['Emergency' 'Trauma' 'Urgent']

Stay values: ['0-10' '41-50' '31-40' '11-20' '51-60' '21-30' '71-80'
 'More than 100 Days' '81-90' '61-70' '91-100']


In [4]:
# Feature Engineering

def stay_to_numeric(stay):
    # WHY: Stay is stored as ranges — we need numbers to calculate averages
    stay_map = {
        '0-10': 5, '11-20': 15, '21-30': 25,
        '31-40': 35, '41-50': 45, '51-60': 55,
        '61-70': 65, '71-80': 75, '81-90': 85,
        '91-100': 95, 'More than 100 Days': 110
    }
    return stay_map.get(stay, np.nan)

df['Stay_Days'] = df['Stay'].apply(stay_to_numeric)

print(f"✓ Stay_Days created")
print(f"Average length of stay: {df['Stay_Days'].mean():.1f} days")
print(f"Median length of stay:  {df['Stay_Days'].median():.1f} days")

✓ Stay_Days created
Average length of stay: 32.1 days
Median length of stay:  25.0 days


In [5]:
# Chart 1, LOS by Department

# WHY: Department-level LoS shows which specialties drive long stays
# Most actionable insight for bed management decisions
dept_los = df.groupby('Department')['Stay_Days'].mean().sort_values(ascending=False).reset_index()
dept_los.columns = ['Department', 'Avg_Stay_Days']
dept_los['Avg_Stay_Days'] = dept_los['Avg_Stay_Days'].round(1)

fig1 = px.bar(
    dept_los, x='Avg_Stay_Days', y='Department',
    orientation='h',
    title='Average Length of Stay by Department',
    labels={'Avg_Stay_Days': 'Average Stay (Days)', 'Department': ''},
    color='Avg_Stay_Days',
    color_continuous_scale='Teal',
)
fig1.update_layout(height=400, showlegend=False,
    coloraxis_showscale=False, plot_bgcolor='white', title_font_size=14)
fig1.write_html('charts/los_by_department.html')
fig1.show()
print(dept_los)

           Department  Avg_Stay_Days
0             surgery           37.6
1        radiotherapy           33.3
2          gynecology           32.2
3  TB & Chest disease           30.9
4          anesthesia           30.1


In [6]:
# Chart 2: Admission type breakdown

# WHY: Emergency vs Elective ratio is a key operational pressure indicator
admission_counts = df['Type of Admission'].value_counts().reset_index()
admission_counts.columns = ['Admission_Type', 'Count']
admission_counts['Percentage'] = (admission_counts['Count'] / len(df) * 100).round(1)

fig2 = px.pie(
    admission_counts, values='Count', names='Admission_Type',
    title='Admission Type Distribution',
    color_discrete_sequence=['#085041','#1D9E75','#9FE1CB'],
    hole=0.4
)
fig2.update_traces(textposition='outside', textinfo='percent+label')
fig2.update_layout(height=400, title_font_size=14)
fig2.write_html('charts/admission_types.html')
fig2.show()
print(admission_counts)


  Admission_Type   Count  Percentage
0         Trauma  152261        47.8
1      Emergency  117676        37.0
2         Urgent   48501        15.2


In [7]:
#Chart 3: LOS by admission type

# WHY: Mean vs Median comparison reveals skew — emergency stays are often more variable
stay_by_admission = df.groupby('Type of Admission')['Stay_Days'].agg(['mean','median']).reset_index()
stay_by_admission.columns = ['Admission_Type', 'Mean_Stay', 'Median_Stay']
stay_by_admission = stay_by_admission.round(1)

fig3 = go.Figure()
fig3.add_trace(go.Bar(name='Mean Stay',
    x=stay_by_admission['Admission_Type'],
    y=stay_by_admission['Mean_Stay'],
    marker_color='#085041'))
fig3.add_trace(go.Bar(name='Median Stay',
    x=stay_by_admission['Admission_Type'],
    y=stay_by_admission['Median_Stay'],
    marker_color='#9FE1CB'))
fig3.update_layout(
    title='Length of Stay by Admission Type — Mean vs Median',
    barmode='group', yaxis_title='Days',
    plot_bgcolor='white', height=400, title_font_size=14)
fig3.write_html('charts/stay_by_admission.html')
fig3.show()
print(stay_by_admission)

  Admission_Type  Mean_Stay  Median_Stay
0      Emergency       30.3         25.0
1         Trauma       34.2         25.0
2         Urgent       30.0         25.0


In [9]:
# Chart 4: Severity Vs LOS

# WHY: Severity should predict LoS — if it doesn't, it suggests triage inconsistencies
severity_los = df.groupby('Severity of Illness')['Stay_Days'].mean().reset_index()
severity_los.columns = ['Severity', 'Avg_Stay_Days']
severity_los['Avg_Stay_Days'] = severity_los['Avg_Stay_Days'].round(1)

severity_order = ['Minor', 'Moderate', 'Extreme']
severity_los['Severity'] = pd.Categorical(severity_los['Severity'],
    categories=severity_order, ordered=True)
severity_los = severity_los.sort_values('Severity')

fig4 = px.bar(severity_los, x='Severity', y='Avg_Stay_Days',
    title='Average Length of Stay by Severity of Illness',
    labels={'Avg_Stay_Days': 'Average Stay (Days)'},
    color='Avg_Stay_Days', color_continuous_scale='Reds',
    text='Avg_Stay_Days')
fig4.update_traces(textposition='outside')
fig4.update_layout(height=400, showlegend=False,
    coloraxis_showscale=False, plot_bgcolor='white', title_font_size=14)
fig4.write_html('charts/stay_by_severity.html')
fig4.show()

In [10]:
#Chart 5: Hospital Benchmarking

# WHY: Comparing hospitals reveals outliers — high LoS hospitals need investigation
# This is benchmarking — a core NHS analytics task
hospital_los = df.groupby('Hospital_code')['Stay_Days'].mean().reset_index()
hospital_los.columns = ['Hospital', 'Avg_Stay_Days']
hospital_los['Avg_Stay_Days'] = hospital_los['Avg_Stay_Days'].round(1)
hospital_los = hospital_los.sort_values('Avg_Stay_Days', ascending=False)

overall_avg = df['Stay_Days'].mean().round(1)

fig5 = px.bar(hospital_los, x='Hospital', y='Avg_Stay_Days',
    title=f'Average Length of Stay by Hospital (Benchmark: {overall_avg} days)',
    labels={'Avg_Stay_Days': 'Average Stay (Days)', 'Hospital': 'Hospital Code'},
    color='Avg_Stay_Days', color_continuous_scale='RdYlGn_r')
fig5.add_hline(y=overall_avg, line_dash='dash', line_color='black',
    annotation_text=f'Average: {overall_avg} days')
fig5.update_layout(height=450, coloraxis_showscale=False,
    plot_bgcolor='white', title_font_size=14)
fig5.write_html('charts/hospital_comparison.html')
fig5.show()

In [12]:
# Exporting full Dashboard HTML

dashboard = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'LoS by Department', 'Admission Type Distribution',
        'LoS by Admission Type', 'LoS by Severity',
        'Hospital Benchmarking', ''
    ],
    specs=[
        [{"type":"bar"}, {"type":"pie"}],
        [{"type":"bar"}, {"type":"bar"}],
        [{"type":"bar","colspan":2}, None]
    ],
    vertical_spacing=0.12
)

for trace in fig1.data:
    trace.marker.color = '#2C5F7A'
    trace.marker.colorscale = None
    dashboard.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    trace.marker.colors = ['#2C5F7A','#5B8FA8','#A8C5D4']
    dashboard.add_trace(trace, row=1, col=2)

for trace in fig3.data:
    if trace.name == 'Mean Stay':
        trace.marker.color = '#2C5F7A'
    else:
        trace.marker.color = '#A8C5D4'
    dashboard.add_trace(trace, row=2, col=1)

for trace in fig4.data:
    trace.marker.color = '#5B8FA8'
    dashboard.add_trace(trace, row=2, col=2)

for trace in fig5.data:
    trace.marker.color = '#2C5F7A'
    dashboard.add_trace(trace, row=3, col=1)

dashboard.update_layout(
    height=1100,
    title_text="Clinical Operations Dashboard — Healthcare Analytics",
    title_font_size=18,
    title_font_color='#1A1A2E',
    showlegend=False,
    plot_bgcolor='#FAFAFA',
    paper_bgcolor='white',
    font=dict(family="Arial", size=11, color='#333333')
)

dashboard.update_xaxes(showgrid=True, gridcolor='#EBEBEB', linecolor='#CCCCCC')
dashboard.update_yaxes(showgrid=True, gridcolor='#EBEBEB', linecolor='#CCCCCC')

dashboard.write_html('charts/clinical_operations_dashboard.html')
print("✓ Dashboard saved with professional colour scheme")

✓ Dashboard saved with professional colour scheme


### Key Findings — Clinical Operations Analysis

### Length of Stay
- Average LoS: check your output — compare against NHS benchmark of ~3-5 days
- Highest LoS departments: TB & Chest Disease, Surgery tend to be highest
- Severity correctly predicts LoS — Extreme cases stay significantly longer than Minor

### Admission Types
- Emergency admissions have higher and more variable stays than elective
- Trauma admissions are most resource-intensive

### Hospital Performance
- Significant variation in LoS across hospitals — outliers warrant clinical audit
- High outlier hospitals: is the longer stay clinically appropriate or a process issue?

### Operational Recommendations
1. Target highest LoS departments for discharge planning improvement
2. Increase elective capacity to reduce emergency bed pressure
3. Audit top 3 outlier hospitals for LoS reduction opportunities
4. Review triage coding — severity should predict LoS more tightly